# Tourism Experience Analytics
## Classification, Prediction, and Recommendation System

This notebook implements data preparation, EDA, rating regression, visit-mode classification, and a content-based attraction recommendation system.

In [ ]:

# ============================================================
# TOURISM EXPERIENCE ANALYTICS
# Classification, Prediction, and Recommendation System
# ============================================================

# 1. Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix
)
from sklearn.metrics.pairwise import cosine_similarity
import warnings
warnings.filterwarnings("ignore")

# 2. Load data
transaction = pd.read_excel("data/Transaction.xlsx")
mode = pd.read_excel("data/Mode.xlsx")
city = pd.read_excel("data/City.xlsx")
country = pd.read_excel("data/Country.xlsx")
user = pd.read_excel("data/User.xlsx")
type_df = pd.read_excel("data/Type.xlsx")
continent = pd.read_excel("data/Continent.xlsx")
region = pd.read_excel("data/Region.xlsx")
item = pd.read_excel("data/Updated_Item.xlsx")

datasets = {
    "Transaction": transaction, "Mode": mode, "City": city,
    "Country": country, "User": user, "Type": type_df,
    "Continent": continent, "Region": region, "Item": item
}
for name, frame in datasets.items():
    print(name, frame.shape)

# 3. Data quality checks
for name, frame in datasets.items():
    print(f"\n{name}: duplicates={frame.duplicated().sum()}")
    print(frame.isna().sum().sum(), "missing cells")

# 4. Clean duplicates
for name in list(datasets):
    datasets[name] = datasets[name].drop_duplicates().copy()

transaction = datasets["Transaction"]
mode = datasets["Mode"]
city = datasets["City"]
country = datasets["Country"]
user = datasets["User"]
type_df = datasets["Type"]
continent = datasets["Continent"]
region = datasets["Region"]
item = datasets["Item"]

# 5. Validate key relationships before merging
print("Invalid UserIds:",
      (~transaction["UserId"].isin(user["UserId"])).sum())
print("Invalid AttractionIds:",
      (~transaction["AttractionId"].isin(item["AttractionId"])).sum())
print("Invalid VisitModeIds:",
      (~transaction["VisitMode"].isin(mode["VisitModeId"])).sum())

# 6. Build consolidated master dataset
u = user.rename(columns={
    "ContinentId":"UserContinentId",
    "RegionId":"UserRegionId",
    "CountryId":"UserCountryId",
    "CityId":"UserCityId"
})
df = transaction.merge(
    u, on="UserId", how="left", validate="many_to_one"
)
df = df.merge(
    mode, left_on="VisitMode", right_on="VisitModeId",
    how="left", validate="many_to_one"
)
df = df.merge(
    item, on="AttractionId", how="left", validate="many_to_one"
)
df = df.merge(
    type_df, on="AttractionTypeId", how="left", validate="many_to_one"
)

city_user = city.rename(columns={
    "CityId":"UserCityId",
    "CityName":"UserCityName",
    "CountryId":"UserCityCountryId"
})
df = df.merge(city_user, on="UserCityId", how="left",
              validate="many_to_one")

city_attr = city.rename(columns={
    "CityId":"AttractionCityId",
    "CityName":"AttractionCityName",
    "CountryId":"AttractionCityCountryId"
})
df = df.merge(city_attr, on="AttractionCityId", how="left",
              validate="many_to_one")

country_user = country.rename(columns={
    "CountryId":"UserCountryId",
    "Country":"UserCountry",
    "RegionId":"UserCountryRegionId"
})
df = df.merge(country_user, on="UserCountryId", how="left",
              validate="many_to_one")

region_user = region.rename(columns={
    "RegionId":"UserRegionId",
    "Region":"UserRegion",
    "ContinentId":"UserRegionContinentId"
})
df = df.merge(region_user, on="UserRegionId", how="left",
              validate="many_to_one")

continent_user = continent.rename(columns={
    "ContinentId":"UserContinentId",
    "Continent":"UserContinent"
})
df = df.merge(continent_user, on="UserContinentId", how="left",
              validate="many_to_one")

df["VisitMode"] = df["VisitMode_y"].fillna(df["VisitMode_x"].astype(str))
df = df[df["Rating"].between(1, 5)].copy()

print("Master dataset:", df.shape)
display(df.head())

# 7. EDA
plt.figure(figsize=(8,5))
df["Rating"].value_counts().sort_index().plot(kind="bar")
plt.title("Distribution of Attraction Ratings")
plt.xlabel("Rating")
plt.ylabel("Number of Transactions")
plt.show()

plt.figure(figsize=(8,5))
df["VisitMode"].value_counts().plot(kind="bar")
plt.title("Visitors by Visit Mode")
plt.xlabel("Visit Mode")
plt.ylabel("Transactions")
plt.xticks(rotation=30)
plt.show()

plt.figure(figsize=(10,5))
df["AttractionType"].value_counts().head(15).plot(kind="bar")
plt.title("Top Attraction Types")
plt.xlabel("Attraction Type")
plt.ylabel("Transactions")
plt.xticks(rotation=45, ha="right")
plt.show()

plt.figure(figsize=(8,5))
df.groupby("VisitYear").size().plot(kind="line", marker="o")
plt.title("Tourism Transactions by Year")
plt.xlabel("Visit Year")
plt.ylabel("Transactions")
plt.grid(alpha=.2)
plt.show()

# 8. Business summaries
print("Top attractions:")
display(df.groupby("Attraction")["Rating"].agg(["count","mean"])
        .sort_values("count", ascending=False).head(10))

print("Average rating by visit mode:")
display(df.groupby("VisitMode")["Rating"].agg(["count","mean"])
        .sort_values("mean", ascending=False))

print("Top user countries:")
display(df["UserCountry"].value_counts().head(10))

# 9. Regression: predict rating
reg_features = [
    "VisitYear","VisitMonth","UserContinent","UserRegion",
    "UserCountry","AttractionType","AttractionCityName"
]
X = df[reg_features].copy()
y = df["Rating"].astype(float)

cat_features = [c for c in reg_features if X[c].dtype == "object"]
num_features = [c for c in reg_features if c not in cat_features]

preprocessor = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_features),
    ("num", "passthrough", num_features)
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=.20, random_state=42
)

reg_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestRegressor(
        n_estimators=50, max_depth=14, min_samples_leaf=3,
        n_jobs=-1, random_state=42
    ))
])
reg_model.fit(X_train, y_train)
reg_pred = reg_model.predict(X_test)

reg_metrics = {
    "MAE": mean_absolute_error(y_test, reg_pred),
    "RMSE": mean_squared_error(y_test, reg_pred) ** 0.5,
    "R2": r2_score(y_test, reg_pred)
}
print("Regression metrics:", reg_metrics)

# 10. Classification: predict visit mode
X = df[reg_features].copy()
y = df["VisitMode"].astype(str)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=.20, random_state=42, stratify=y
)

clf_preprocessor = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_features),
    ("num", "passthrough", num_features)
])

clf_model = Pipeline([
    ("preprocessor", clf_preprocessor),
    ("model", RandomForestClassifier(
        n_estimators=60, max_depth=16, min_samples_leaf=3,
        n_jobs=-1, random_state=42,
        class_weight="balanced_subsample"
    ))
])
clf_model.fit(X_train, y_train)
clf_pred = clf_model.predict(X_test)

clf_metrics = {
    "Accuracy": accuracy_score(y_test, clf_pred),
    "Precision_weighted": precision_score(y_test, clf_pred, average="weighted", zero_division=0),
    "Recall_weighted": recall_score(y_test, clf_pred, average="weighted", zero_division=0),
    "F1_weighted": f1_score(y_test, clf_pred, average="weighted", zero_division=0)
}
print("Classification metrics:", clf_metrics)
print(classification_report(y_test, clf_pred, zero_division=0))

# 11. Recommendation system
# Attraction-level profile: average rating, popularity, city, type.
attr = df.groupby(
    ["AttractionId","Attraction","AttractionCityName","AttractionType"],
    dropna=False
).agg(
    AvgRating=("Rating","mean"),
    RatingCount=("Rating","count")
).reset_index()

# Use type and city as content features.
content = pd.get_dummies(
    attr[["AttractionCityName","AttractionType"]].fillna("Unknown")
).astype(float)

# Add scaled rating/popularity signals.
content["AvgRating"] = attr["AvgRating"].fillna(attr["AvgRating"].mean()) / 5.0
content["Popularity"] = (
    np.log1p(attr["RatingCount"]) /
    np.log1p(attr["RatingCount"]).max()
)

similarity = cosine_similarity(content.values)

def recommend_attractions(attraction_id=None, attraction_type=None, city_name=None, top_n=5):
    if attraction_id is not None and attraction_id in set(attr["AttractionId"]):
        idx = attr.index[attr["AttractionId"].eq(attraction_id)][0]
        scores = similarity[idx].copy()
        scores[idx] = -1
        order = np.argsort(scores)[::-1]
        return attr.iloc[order].head(top_n)[
            ["Attraction","AttractionCityName","AttractionType","AvgRating","RatingCount"]
        ]

    candidates = attr.copy()
    if attraction_type:
        candidates = candidates[candidates["AttractionType"].eq(attraction_type)]
    if city_name:
        candidates = candidates[candidates["AttractionCityName"].eq(city_name)]
    return candidates.sort_values(
        ["AvgRating","RatingCount"], ascending=[False,False]
    ).head(top_n)[
        ["Attraction","AttractionCityName","AttractionType","AvgRating","RatingCount"]
    ]

display(recommend_attractions(top_n=5))

# 12. Save final artifacts
df.to_csv("tourism_master_dataset.csv", index=False)
joblib.dump(reg_model, "rating_regressor.joblib")
joblib.dump(clf_model, "visit_mode_classifier.joblib")
print("Artifacts saved successfully.")
